# 미국 언론사 RSS 현재 시점 수집 조사

`2_rss_history_test.ipynb`에서 **과거 기사로는 접근할 수 없어 제외한 피드(`LIVE_ONLY`)** 중,
지금부터 주기적으로 받아 쌓는 방식(지속 수집)에 쓸 수 있는 피드를 고르는 노트북입니다.

**확인하는 것**
1. 신뢰도·금융 관련성: 전문 언론사인지, 증권·금융 쪽에서 참고할 만한지 (언론사 단위 분류 `SOURCE_TIERS`)
2. 현재 RSS 응답과 갱신 속도: 하루 기사 수, 피드 하나가 담는 시간 → 기사를 놓치지 않으려면 몇 분 간격으로 받아야 하는지
3. 본문 확보: 지금 올라온 기사로 RSS 본문과 원문 페이지 본문을 추출·대조 (`2_rss_history_test.ipynb` 7단계와 같은 함수)
4. 팀원 통합 수집기(`team_rss_news/src/rss_pipeline.py`)에 이미 있는 피드인지

**결과 변수**: `LIVE_COLLECT_FEEDS`(본문까지 수집 가능), `LIVE_UNSTABLE_FEEDS`, `LIVE_RSS_UNVERIFIED_FEEDS`(RSS 본문만, 원문 대조 불가), `LIVE_LIST_ONLY_FEEDS`(목록만, 본문 불가),
`LIVE_EXCLUDED_FEEDS`(신뢰도·관련성으로 제외), `IN_TEAM_PIPELINE_FEEDS`(팀원 수집기에 이미 있음)

팀원 수집기 설정(`feed_registry.csv`) 형식으로 추가 후보를 `rss_live/registry_additions.csv`에 저장합니다.

In [1]:
import calendar
import gzip
import hashlib
import html
import json
import re
import time
import unicodedata
from datetime import date, datetime, timedelta, timezone
from pathlib import Path
from urllib.parse import urlparse
from zoneinfo import ZoneInfo

import feedparser
import pandas as pd
import requests
from dateutil import parser as date_parser
from lxml import html as lxml_html
from trafilatura import extract as trafilatura_extract
from trafilatura.utils import decode_file

## 0. 설정

In [2]:
# ============================================================
# 경로
# ============================================================
BASE_DIR = Path("rss_live")
CACHE_DIR = BASE_DIR / "cache"
FEED_CACHE_DIR = CACHE_DIR / "feeds"

# 과거 수집 테스트 결과 (LIVE_ONLY 피드 목록)
PAST_FEED_LISTS_PATH = Path("rss_history/feed_lists.json")

# 팀원 통합 수집기 (rss_news). 이미 수집 중인 피드를 확인하는 데만 사용
TEAM_RSS_DIR = Path("team_rss_news")

LIVE_CHECK_PATH = BASE_DIR / "live_check.csv"
LIVE_SUMMARY_PATH = BASE_DIR / "live_summary.csv"
LIVE_LISTS_PATH = BASE_DIR / "live_feed_lists.json"
REGISTRY_EXPORT_PATH = BASE_DIR / "registry_additions.csv"

# ============================================================
# 요청 설정 (2_rss_history_test.ipynb와 같음)
# ============================================================
NEWS_TIMEZONE = ZoneInfo("America/New_York")
REQUEST_TIMEOUT = 30
REQUEST_DELAY = 0.5
MAX_RETRIES = 3
WAYBACK_BLOCK_WAIT = 60
MIN_VALID_DATE = datetime(1995, 1, 1, tzinfo=timezone.utc)

REQUEST_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0 Safari/537.36"
    ),
    "Accept": "application/rss+xml, application/atom+xml, application/xml;q=0.9, */*;q=0.8",
}

# ============================================================
# 갱신 속도 판정
# ============================================================
STALE_DAYS = 7              # 최신 기사가 이보다 오래되면 갱신 멈춤
RATE_WINDOW_HOURS = 24      # 최근 24시간 기사 수로 시간당 발행량 계산

# 피드 하나가 담는 시간(= 피드 기사 수 / 시간당 발행량)의 이 비율 간격으로 받으면 앞뒤가 겹쳐서 누락이 없음
POLL_SAFETY = 0.5
MIN_POLL_MINUTES = 5        # 이보다 자주 받아야 하는 피드는 '놓칠 위험 큼'으로 표시
MAX_POLL_MINUTES = 360      # 발행량이 적어도 6시간에 한 번은 받음

# ============================================================
# 본문 테스트
# ============================================================
LIVE_SAMPLE_PER_SOURCE = 8  # 언론사별 현재 기사 표본 수

In [3]:
# ============================================================
# 테스트 후보 피드
#   feed_id: {source: 도메인, name: 언론사명, section: 섹션, url: 피드 URL}
#
# - 언론사 하나가 섹션 피드를 여러 개 제공하면 섹션별로 따로 둡니다.
# - WordPress 사이트는 사이트 전체 피드(/feed/)여야 날짜 아카이브 테스트 대상이 됩니다.
# - 2026-11-04 사전 조사에서 이미 응답이 없던 피드도 기록을 위해 포함 (1단계에서 걸러짐)
# ============================================================
def feed(source, name, section, url):
    return {"source": source, "name": name, "section": section, "url": url}


CNBC = "https://search.cnbc.com/rs/search/combinedcms/view.xml?partnerId=wrss01&id="
DJ = "https://feeds.content.dowjones.io/public/rss/"

CANDIDATE_FEEDS = {
    # ── 미국 증시/금융 전문 ──
    "cnbc_top": feed("cnbc.com", "CNBC", "Top News", CNBC + "100003114"),
    "cnbc_business": feed("cnbc.com", "CNBC", "Business", CNBC + "10001147"),
    "cnbc_investing": feed("cnbc.com", "CNBC", "Investing", CNBC + "15839069"),
    "cnbc_market_insider": feed("cnbc.com", "CNBC", "Market Insider", CNBC + "20409666"),
    "cnbc_economy": feed("cnbc.com", "CNBC", "Economy", CNBC + "20910258"),
    "cnbc_earnings": feed("cnbc.com", "CNBC", "Earnings", CNBC + "15839135"),
    "cnbc_finance": feed("cnbc.com", "CNBC", "Finance", CNBC + "10000664"),
    "cnbc_tech": feed("cnbc.com", "CNBC", "Tech", CNBC + "19854910"),

    "wsj_markets": feed("wsj.com", "Wall Street Journal", "Markets", DJ + "RSSMarketsMain"),
    "wsj_business": feed("wsj.com", "Wall Street Journal", "US Business", DJ + "WSJcomUSBusiness"),
    "wsj_economy": feed("wsj.com", "Wall Street Journal", "Economy", DJ + "socialeconomyfeed"),
    "wsj_tech": feed("wsj.com", "Wall Street Journal", "Technology", DJ + "RSSWSJD"),
    "wsj_world": feed("wsj.com", "Wall Street Journal", "World", DJ + "RSSWorldNews"),

    "marketwatch_top": feed("marketwatch.com", "MarketWatch", "Top Stories", DJ + "mw_topstories"),
    "marketwatch_bulletins": feed("marketwatch.com", "MarketWatch", "Breaking News", DJ + "mw_bulletins"),
    "marketwatch_pulse": feed("marketwatch.com", "MarketWatch", "MarketPulse", DJ + "mw_marketpulse"),

    "barrons": feed("barrons.com", "Barron's", "All", "https://www.barrons.com/feed"),
    "bloomberg_markets": feed("bloomberg.com", "Bloomberg", "Markets", "https://feeds.bloomberg.com/markets/news.rss"),
    "bloomberg_economics": feed("bloomberg.com", "Bloomberg", "Economics", "https://feeds.bloomberg.com/economics/news.rss"),
    "bloomberg_tech": feed("bloomberg.com", "Bloomberg", "Technology", "https://feeds.bloomberg.com/technology/news.rss"),

    "ibd": feed("investors.com", "Investor's Business Daily", "All", "https://www.investors.com/feed/"),
    "seekingalpha_all": feed("seekingalpha.com", "Seeking Alpha", "All Articles", "https://seekingalpha.com/feed.xml"),
    "seekingalpha_currents": feed("seekingalpha.com", "Seeking Alpha", "Market Currents", "https://seekingalpha.com/market_currents.xml"),
    "fool": feed("fool.com", "The Motley Fool", "All", "https://www.fool.com/a/feeds/foolwatch?apikey=foolwatch-feed"),
    "nasdaq_markets": feed("nasdaq.com", "Nasdaq", "Markets", "https://www.nasdaq.com/feed/rssoutbound?category=Markets"),
    "investing_news": feed("investing.com", "Investing.com", "All News", "https://www.investing.com/rss/news.rss"),
    "investing_economy": feed("investing.com", "Investing.com", "Economy", "https://www.investing.com/rss/news_14.rss"),
    "marketbeat": feed("marketbeat.com", "MarketBeat", "All", "https://www.marketbeat.com/feed/"),
    "thestreet": feed("thestreet.com", "TheStreet", "All", "https://www.thestreet.com/.rss/full/"),
    "kiplinger": feed("kiplinger.com", "Kiplinger", "All", "https://www.kiplinger.com/feeds.xml"),
    "yahoo_finance": feed("finance.yahoo.com", "Yahoo Finance", "All", "https://finance.yahoo.com/news/rssindex"),
    "zacks": feed("zacks.com", "Zacks", "All", "https://scr.zacks.com/rss"),

    # ── 경제/비즈니스 종합 ──
    "fortune": feed("fortune.com", "Fortune", "All", "https://fortune.com/feed/"),
    "business_insider": feed("businessinsider.com", "Business Insider", "All", "https://feeds.businessinsider.com/custom/all"),
    "foxbusiness_latest": feed("foxbusiness.com", "Fox Business", "Latest", "https://moxie.foxbusiness.com/google-publisher/latest.xml"),
    "foxbusiness_markets": feed("foxbusiness.com", "Fox Business", "Markets", "https://moxie.foxbusiness.com/google-publisher/markets.xml"),
    "foxbusiness_economy": feed("foxbusiness.com", "Fox Business", "Economy", "https://moxie.foxbusiness.com/google-publisher/economy.xml"),
    "cbs_moneywatch": feed("cbsnews.com", "CBS News", "MoneyWatch", "https://www.cbsnews.com/latest/rss/moneywatch"),
    "nypost": feed("nypost.com", "New York Post", "All", "https://nypost.com/feed/"),
    "nypost_business": feed("nypost.com", "New York Post", "Business", "https://nypost.com/business/feed/"),
    "usatoday_money": feed("usatoday.com", "USA Today", "Money", "https://rssfeeds.usatoday.com/UsatodaycomMoney-TopStories"),
    "cnn_money": feed("cnn.com", "CNN Business", "Money", "http://rss.cnn.com/rss/money_latest.rss"),
    "forbes_business": feed("forbes.com", "Forbes", "Business", "https://www.forbes.com/business/feed/"),
    "forbes_innovation": feed("forbes.com", "Forbes", "Innovation", "https://www.forbes.com/innovation/feed/"),
    "fastcompany": feed("fastcompany.com", "Fast Company", "Latest", "https://www.fastcompany.com/latest/rss"),
    "qz": feed("qz.com", "Quartz", "All", "https://qz.com/rss"),
    "axios": feed("axios.com", "Axios", "All", "https://api.axios.com/feed/"),
    "semafor": feed("semafor.com", "Semafor", "All", "https://www.semafor.com/rss.xml"),

    # ── 종합 일간지/방송 ──
    "nyt_business": feed("nytimes.com", "New York Times", "Business", "https://rss.nytimes.com/services/xml/rss/nyt/Business.xml"),
    "nyt_economy": feed("nytimes.com", "New York Times", "Economy", "https://rss.nytimes.com/services/xml/rss/nyt/Economy.xml"),
    "nyt_tech": feed("nytimes.com", "New York Times", "Technology", "https://rss.nytimes.com/services/xml/rss/nyt/Technology.xml"),
    "nyt_home": feed("nytimes.com", "New York Times", "Home Page", "https://rss.nytimes.com/services/xml/rss/nyt/HomePage.xml"),
    "wapo_business": feed("washingtonpost.com", "Washington Post", "Business", "https://feeds.washingtonpost.com/rss/business"),
    "latimes_business": feed("latimes.com", "Los Angeles Times", "Business", "https://www.latimes.com/business/rss2.0.xml"),
    "guardian_us_business": feed("theguardian.com", "The Guardian", "US Business", "https://www.theguardian.com/us/business/rss"),
    "npr_business": feed("npr.org", "NPR", "Business", "https://feeds.npr.org/1006/rss.xml"),
    "npr_economy": feed("npr.org", "NPR", "Economy", "https://feeds.npr.org/1017/rss.xml"),
    "abc_business": feed("abcnews.go.com", "ABC News", "Business", "https://abcnews.go.com/abcnews/moneyheadlines"),
    "nbc_business": feed("nbcnews.com", "NBC News", "Business", "https://feeds.nbcnews.com/nbcnews/public/business"),
    "apnews": feed("apnews.com", "Associated Press", "All", "https://apnews.com/index.rss"),
    "yahoo_news": feed("yahoo.com", "Yahoo News", "All", "https://news.yahoo.com/rss/"),
    "politico": feed("politico.com", "Politico", "Politics", "https://rss.politico.com/politics-news.xml"),
    "thehill": feed("thehill.com", "The Hill", "All", "https://thehill.com/feed/"),
    "time": feed("time.com", "TIME", "All", "https://time.com/feed/"),
    "washington_examiner": feed("washingtonexaminer.com", "Washington Examiner", "All", "https://www.washingtonexaminer.com/feed/"),
    "nationalreview": feed("nationalreview.com", "National Review", "All", "https://www.nationalreview.com/feed/"),
    "motherjones": feed("motherjones.com", "Mother Jones", "All", "https://www.motherjones.com/feed/"),

    # ── IT/반도체 ──
    "techcrunch": feed("techcrunch.com", "TechCrunch", "All", "https://techcrunch.com/feed/"),
    "theverge": feed("theverge.com", "The Verge", "All", "https://www.theverge.com/rss/index.xml"),
    "arstechnica": feed("arstechnica.com", "Ars Technica", "All", "https://arstechnica.com/feed/"),
    "wired": feed("wired.com", "WIRED", "All", "https://www.wired.com/feed/rss"),
    "engadget": feed("engadget.com", "Engadget", "All", "https://www.engadget.com/rss.xml"),
    "cnet": feed("cnet.com", "CNET", "News", "https://www.cnet.com/rss/news/"),
    "9to5mac": feed("9to5mac.com", "9to5Mac", "All", "https://9to5mac.com/feed/"),
    "tomshardware": feed("tomshardware.com", "Tom's Hardware", "All", "https://www.tomshardware.com/feeds.xml"),
    "eetimes": feed("eetimes.com", "EE Times", "All", "https://www.eetimes.com/feed/"),
    "semiengineering": feed("semiengineering.com", "Semiconductor Engineering", "All", "https://semiengineering.com/feed/"),
    "venturebeat": feed("venturebeat.com", "VentureBeat", "All", "https://venturebeat.com/feed/"),

    # ── 보도자료 배포 ──
    "businesswire": feed("businesswire.com", "Business Wire", "All", "https://feed.businesswire.com/rss/home/?rss=G1QFDERJXkJeEFtRWA=="),
    "prnewswire": feed("prnewswire.com", "PR Newswire", "All", "https://www.prnewswire.com/rss/news-releases-list.rss"),
    "globenewswire": feed("globenewswire.com", "GlobeNewswire", "Public Companies", "https://www.globenewswire.com/RssFeed/orgclass/1/feedTitle/GlobeNewswire%20-%20News%20about%20Public%20Companies"),
}

# 테스트 전에 내용을 보고 제외한 피드와 이유
EXCLUDED_FEEDS = {
    # /feed 가 뉴스가 아니라 재테크 가이드(benzinga.com/money) 글 목록이고,
    # 1페이지 기사 시각이 1초 간격으로 일괄 갱신돼 있어 작성 시각으로 쓸 수 없음
    "benzinga": "재테크 가이드 피드 (뉴스 아님, 작성 시각 신뢰 불가)",
}

print(f"후보 피드 {len(CANDIDATE_FEEDS)}개 / 언론사 {len({f['source'] for f in CANDIDATE_FEEDS.values()})}곳")

후보 피드 81개 / 언론사 56곳


In [4]:
# ============================================================
# 본문 수집 설정 (7~9단계)
# ============================================================
# 통과 피드마다 테스트 기간 기사 중 이 수만큼 무작위 표본 (기사가 적으면 전부)
BODY_SAMPLE_PER_FEED = 40
BODY_SAMPLE_SEED = 42

# 기사 페이지는 피드보다 요청 간격을 넉넉히
BODY_FETCH_DELAY = 1.0
BODY_FETCH_RETRIES = 1

# 같은 사이트에서 연속으로 이 횟수만큼 차단(401/403/429)·연결 오류가 나면 그 사이트 나머지 기사는 요청하지 않음
BODY_HOST_FAIL_LIMIT = 3

HTML_HEADERS = {
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
}

MIN_BODY_WORDS = 80         # 본문 최소 단어 수 (짧은 단신도 통과하도록 낮게)
MIN_LD_COVERAGE = 0.6       # JSON-LD articleBody 대비 추출 본문 비율 하한
SHINGLE_SIZE = 5            # 두 본문을 비교하는 단위 (연속 5단어 묶음)
MIN_REF_COVERAGE = 0.8      # 기준 본문(JSON-LD 또는 사이트별 본문 영역)의 80% 미만만 trafilatura에 있으면 기준 본문 사용
RSS_FULL_COVERAGE = 0.9     # 원문 본문의 90% 이상이 RSS 본문에 있으면 RSS 본문 = 전문

# 사이트별 본문 영역 (태그, class 이름). trafilatura 결과를 교차 확인하는 기준 본문으로 사용
#   2026-11-05 각 사이트 기사 2건씩 HTML을 열어 <p>가 모여 있는 요소를 확인해서 정함
#   (rss_news 본문 시험 코드의 매체별 SELECTORS와 같은 방식)
#   등록된 사이트는 이 영역을 최종 본문으로 쓰고, trafilatura 결과와 겹침 비율로 서로 확인
SITE_BODY_CONTAINERS = {
    "motherjones.com": ("article", "entry-content"),
    "techcrunch.com": ("div", "entry-content"),
    "semiengineering.com": ("div", "post_cnt"),
    "nypost.com": ("div", "single__content"),
    "arstechnica.com": ("div", "post-content"),   # 광고를 사이에 두고 여러 개로 나뉨 → trafilatura가 첫 덩어리만 가져감
    "9to5mac.com": ("div", "post-content"),
    "wired.com": ("div", "body__inner-container"),
}

# 본문 영역 안에서 빼는 요소 (class 단어가 이 접두어로 시작)
JUNK_CLASS_PREFIXES = (
    "ad-", "ads", "advert", "newsletter", "related", "promo", "recirc", "share", "social", "subscribe",
    "inline-module",   # NY Post 본문 중간 뉴스레터 가입·관련 기사 박스
)

# 본문 영역 문단 중 링크 글자가 이 비율 이상이면 관련 기사·상품 링크 목록으로 보고 제외 (9to5Mac 쇼핑 목록 등)
MAX_LINK_DENSITY = 0.8

MIN_BODY_ARTICLES = 8       # 판정에 필요한 최소 기사 페이지 수
MIN_RSS_COMPARABLE = 5      # RSS 전문 여부 판정에 필요한 최소 비교 가능 기사 수
BODY_USABLE_RATE = 0.8
BODY_UNSTABLE_RATE = 0.5

HTML_DIR = CACHE_DIR / "html"
HTML_FETCH_LOG_PATH = CACHE_DIR / "html_fetch.jsonl"

BODY_CHECK_PATH = BASE_DIR / "body_check.parquet"
BODY_SUMMARY_PATH = BASE_DIR / "body_summary.csv"
BODY_REVIEW_PATH = BASE_DIR / "body_review.csv"
BODY_LISTS_PATH = BASE_DIR / "body_lists.json"

In [5]:
# ============================================================
# 페이지 판정 패턴 (1_google_rss_source_test.ipynb와 같은 기준)
# ============================================================
VIDEO_PATH_PATTERN = re.compile(
    r"/(video|videos|watch|clip|clips|v|live-tv|livetv|podcast|podcasts|audio|gallery|galleries|slideshow|photos)(/|$)",
    re.IGNORECASE,
)
VIDEO_HOST_PATTERN = re.compile(r"^(video|videos|watch|tv)\.", re.IGNORECASE)
LIVE_BLOG_PATH_PATTERN = re.compile(r"/(live|live-news|live-updates|liveblog)(/|$)|live-updates", re.IGNORECASE)

ARTICLE_LD_TYPES = {
    "Article", "NewsArticle", "ReportageNewsArticle", "AnalysisNewsArticle", "OpinionNewsArticle",
    "BackgroundNewsArticle", "ReviewNewsArticle", "BlogPosting", "Report",
    "Review",   # WIRED 제품 리뷰 (og:type은 website)
}

PAYWALL_PHRASES = [
    "subscribe to continue", "subscribe to read", "subscription required", "to continue reading",
    "already a subscriber", "sign in to continue", "sign in to read", "log in to continue",
    "create a free account to continue", "this article is for subscribers", "this content is for subscribers",
    "unlock this article", "become a member to read",
]

# 봇 차단 / 챌린지 페이지 흔적 (<title> 또는 HTML 원문)
# 뒤의 세 문구는 rss_news 본문 시험에서 Zacks·MarketWatch·WSJ 차단 페이지로 확인된 문구
BLOCK_TITLE_PHRASES = [
    "access denied", "just a moment", "attention required", "are you a robot", "pardon our interruption",
    "request unsuccessful", "verify you are human", "403 forbidden",
]
BLOCK_HTML_MARKERS = [
    "px-captcha", "captcha-delivery.com", "cf-chl-", "_incapsula_resource",
    "made us think you were a bot", "please enable js and disable any ad blocker",
]

PUBLISHED_DATE_FIELDS = [
    "ld:datePublished", "property:article:published_time", "itemprop:datePublished",
    "name:parsely-pub-date", "name:pubdate", "name:publish-date", "name:sailthru.date",
]

# 언론사별 제외 구역: "호스트/경로" 문자열(www. 제거)에 정규식 검색 (1_google_rss_source_test.ipynb의 SECTION_EXCLUDE_RULES)
# URL만으로 판단하므로 다운로드 전에 거름
SECTION_EXCLUDE_RULES = {
    "nypost.com": [
        r"/betting/",     # 도박 프로모션 (테스트 기간 NY Post 505건 중 25건)
    ],
}

# 아래 판정은 URL·메타데이터로 걸러낼 수 있는 기사 아닌 페이지 → 성공률 분모에서 제외
FILTERABLE_STATUSES = {"EXCLUDED_SECTION", "VIDEO_PAGE", "LIVE_BLOG", "NOT_ARTICLE"}

# ============================================================
# RSS 본문 처리 패턴
# ============================================================
# RSS 본문 끝에 붙는 출처 문구 (기사 본문이 아니므로 제거)
#   WordPress 기본 "The post ... appeared first on ...", Fortune "This story was originally featured on ..." (rss_news 감사에서 확인)
RSS_FOOTER_PATTERN = re.compile(
    r"^(the post .+ appeared first on .+|this (story|article) (was )?originally (appeared|featured|published) .+)$",
    re.IGNORECASE,
)

# RSS 본문이 티저(앞부분만)로 끝나는 흔적. rss_news 감사에서 The Motley Fool이 "Continue reading"으로 끝나는 사례 확인
RSS_TEASER_PATTERN = re.compile(r"(continue reading|read more|keep reading|\[…\]|\[\.\.\.\]|…|\.\.\.)\W*$", re.IGNORECASE)

# 추출한 본문에 섞이면 안 되는 문구 (줄 단위 검사, 검증용)
#   rss_news 매체 검토에서 정제 필요로 기록된 유형: 뉴스레터·앱 홍보, 추천/관련 기사, 출처 푸터
BODY_JUNK_PATTERNS = {
    "newsletter": re.compile(r"sign up for .{0,40}newsletter|subscribe to (our|the) .{0,30}newsletter", re.IGNORECASE),
    "app_promo": re.compile(r"download the .{0,30}app", re.IGNORECASE),
    "related": re.compile(r"^(read more|related|recommended|more from|see also|most popular|trending)\b.{0,60}$", re.IGNORECASE),
    "advertisement": re.compile(r"^advertisement$", re.IGNORECASE),
    "rss_footer": re.compile(r"appeared first on|originally (appeared|featured) on", re.IGNORECASE),
}

In [6]:
# 현재 시점 조사용으로 바꾼 값·추가한 값
MIN_BODY_ARTICLES = 5       # 언론사별 표본이 8건이라 판정 최소 기사 수를 낮춤
MIN_REF_PRECISION = 0.9     # trafilatura 본문 중 JSON-LD 본문에 있는 비율이 이보다 낮으면 JSON-LD 본문 사용

# RSS 본문만 있는 경우 (원문 차단): RSS 본문이 80단어 이상인 기사 비율이 이 이상이고 티저가 드물면 'RSS 본문(미검증)'
RSS_ONLY_MIN_RATE = 0.8
RSS_ONLY_MAX_TEASER = 0.2

# 언론사가 붙인 AI 생성 요약 (LA Times "Perspectives")
BODY_JUNK_PATTERNS["ai_generated"] = re.compile(r"ai-generated content|generated by ai", re.IGNORECASE)

### 0-1. 신뢰도·금융 관련성 분류

언론사 단위로 나눕니다. **제 판단으로 만든 초안이므로 검토 후 조정하세요.**

| 등급 | 의미 | 수집 |
|---|---|---|
| `FINANCE_CORE` | 경제·증시 전문 언론 | 수집 |
| `GENERAL_MAJOR` | 주요 종합 일간지·방송·정책 전문지 (경제·정책 기사) | 수집 |
| `TECH` | 기술 전문지 (반도체·빅테크 참고용) | 수집 (보조) |
| `PRESS_RELEASE` | 기업 보도자료 배포 (1차 자료, 기사 아님) | 별도 용도로 수집 |
| `EXCLUDE` | 타블로이드·의견·개인 재테크·자동 생성·타사 기사 모음·제품 리뷰 위주 | 제외 |

In [7]:
# {도메인: (등급, 이유)}
SOURCE_TIERS = {
    # 경제·증시 전문
    "cnbc.com": ("FINANCE_CORE", "미국 경제·증시 전문 방송"),
    "wsj.com": ("FINANCE_CORE", "미국 대표 경제지 (Dow Jones)"),
    "marketwatch.com": ("FINANCE_CORE", "Dow Jones 증시 뉴스"),
    "bloomberg.com": ("FINANCE_CORE", "금융 전문 통신·방송"),
    "investing.com": ("FINANCE_CORE", "시장 속보·경제지표 뉴스 (자체 기사 + 통신 전재)"),
    "fortune.com": ("FINANCE_CORE", "경제·기업 전문지"),
    "businessinsider.com": ("FINANCE_CORE", "경제·기업 뉴스"),
    "foxbusiness.com": ("FINANCE_CORE", "경제·증시 방송"),

    # 주요 종합지·방송·정책
    "nytimes.com": ("GENERAL_MAJOR", "미국 대표 일간지"),
    "latimes.com": ("GENERAL_MAJOR", "주요 일간지"),
    "theguardian.com": ("GENERAL_MAJOR", "주요 일간지"),
    "npr.org": ("GENERAL_MAJOR", "공영 방송"),
    "abcnews.go.com": ("GENERAL_MAJOR", "주요 방송"),
    "nbcnews.com": ("GENERAL_MAJOR", "주요 방송"),
    "cbsnews.com": ("GENERAL_MAJOR", "주요 방송"),
    "axios.com": ("GENERAL_MAJOR", "정책·경제 속보 전문"),
    "semafor.com": ("GENERAL_MAJOR", "정책·경제 뉴스"),
    "politico.com": ("GENERAL_MAJOR", "정책·정치 전문지 (관세·규제 등 정책 변수)"),
    "time.com": ("GENERAL_MAJOR", "주요 시사 주간지"),

    # 기술 전문 (반도체·빅테크 참고)
    "theverge.com": ("TECH", "기술 전문지"),
    "engadget.com": ("TECH", "기술 전문지"),
    "tomshardware.com": ("TECH", "PC·반도체 하드웨어 전문 (리뷰·할인 기사 섞임)"),
    "venturebeat.com": ("TECH", "AI·엔터프라이즈 기술 전문"),

    # 보도자료
    "businesswire.com": ("PRESS_RELEASE", "기업 보도자료 배포 (실적·공시성 발표)"),
    "prnewswire.com": ("PRESS_RELEASE", "기업 보도자료 배포 (실적·공시성 발표)"),

    # 제외
    "seekingalpha.com": ("EXCLUDE", "개인 투자자 기고 위주"),
    "fool.com": ("EXCLUDE", "개인 투자 조언·유료 서비스 홍보"),
    "nasdaq.com": ("EXCLUDE", "타사 투자 기사(Zacks, Motley Fool 등) 재게재 위주"),
    "marketbeat.com": ("EXCLUDE", "데이터 기반 자동 생성 기사"),
    "thestreet.com": ("EXCLUDE", "개인 투자 조언 위주"),
    "kiplinger.com": ("EXCLUDE", "개인 재테크"),
    "forbes.com": ("EXCLUDE", "외부 기고자(contributor) 글 비중 큼"),
    "nypost.com": ("EXCLUDE", "타블로이드"),
    "qz.com": ("EXCLUDE", "2024년부터 AI 생성 기사 게재"),
    "washingtonexaminer.com": ("EXCLUDE", "당파성 강한 논조"),
    "yahoo.com": ("EXCLUDE", "타사 기사 모음"),
    "cnet.com": ("EXCLUDE", "소비자 제품 리뷰·구매 가이드 위주"),
}

COLLECT_TIERS = ["FINANCE_CORE", "GENERAL_MAJOR", "TECH", "PRESS_RELEASE"]


def tier_of(info):
    return SOURCE_TIERS.get(info["source"], ("UNCLASSIFIED", "분류 안 됨"))

## 0-2. 공통 유틸 (2_rss_history_test.ipynb와 같음)

In [8]:
FEED_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# 캐시 폴더는 git에 올리지 않음
(CACHE_DIR / ".gitignore").write_text("*\n")


def append_jsonl(path, record):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def load_jsonl(path):
    path = Path(path)

    if not path.exists():
        return []

    records = []

    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if line:
                records.append(json.loads(line))

    return records


def now_utc():
    return datetime.now(timezone.utc)


def url_hash(url):
    return hashlib.sha1(url.encode("utf-8")).hexdigest()


def et_day_bounds(start, end):
    # 동부시간 날짜 구간 [start, end] → UTC [시작, 끝)
    start_dt = datetime.combine(start, datetime.min.time(), NEWS_TIMEZONE)
    end_dt = datetime.combine(end + timedelta(days=1), datetime.min.time(), NEWS_TIMEZONE)

    return start_dt.astimezone(timezone.utc), end_dt.astimezone(timezone.utc)


def date_range(start, end):
    days = []
    current = start

    while current <= end:
        days.append(current)
        current += timedelta(days=1)

    return days


def to_feed_dict(feed_ids, extra=None):
    # feed_id 리스트 → {feed_id: 후보 정보 (+추가 정보)} (CANDIDATE_FEEDS 순서 유지)
    feed_ids = set(feed_ids)
    extra = extra or {}

    return {
        feed_id: {**info, **extra.get(feed_id, {})}
        for feed_id, info in CANDIDATE_FEEDS.items()
        if feed_id in feed_ids
    }


def make_session():
    session = requests.Session()
    session.headers.update(REQUEST_HEADERS)

    return session


SESSION = make_session()
_last_request_at = {}


def polite_get(url, delay=REQUEST_DELAY, headers=None, retries=MAX_RETRIES):
    # 같은 호스트에 delay초 간격을 두고 요청, 429/5xx/네트워크 오류는 retries번까지 재시도
    # headers: 기본 헤더(RSS용)에 덮어쓸 헤더 (기사 HTML 요청은 HTML_HEADERS)
    host = urlparse(url).hostname

    for attempt in range(retries + 1):
        wait = _last_request_at.get(host, 0) + delay - time.time()

        if wait > 0:
            time.sleep(wait)

        try:
            # archive.org는 Accept에 RSS 형식을 지정하면 요청 제한과 상관없이 바로 429를 돌려줌
            if host == "web.archive.org":
                headers = {"Accept": "*/*"}

            response = SESSION.get(url, timeout=REQUEST_TIMEOUT, headers=headers)
            error = None

        except requests.RequestException as e:
            response = None
            error = f"{type(e).__name__}: {e}"

        finally:
            _last_request_at[host] = time.time()

        retryable = response is None or response.status_code in {429, 500, 502, 503, 504}

        if not retryable or attempt == retries:
            return response, error

        if host == "web.archive.org":
            time.sleep(WAYBACK_BLOCK_WAIT * (attempt + 1))
        else:
            time.sleep(5 * (attempt + 1))


def entry_published(entry):
    # feedparser가 UTC로 정규화한 시각 (published → updated 순)
    for key in ("published_parsed", "updated_parsed"):
        value = entry.get(key)

        if value:
            published = datetime.fromtimestamp(calendar.timegm(value), timezone.utc)
            return published if published >= MIN_VALID_DATE else None

    return None


def parse_feed(content):
    parsed = feedparser.parse(content)
    entries = []

    for entry in parsed.entries:
        published_utc = entry_published(entry)

        entries.append({
            # Atom 피드(The Hill 등)는 제목에 &#8216; 같은 엔티티가 남아 있어 한 번 더 해제
            "title": html.unescape(entry.get("title") or "").strip(),
            "link": entry.get("link"),
            "guid": entry.get("id") or entry.get("link"),
            "published_utc": published_utc,
            "published_et": published_utc.astimezone(NEWS_TIMEZONE) if published_utc else None,
            "summary": html.unescape(entry.get("summary") or "").strip(),
            # RSS 본문 HTML (content:encoded / Atom content). 제공하지 않는 피드는 빈 문자열
            "content": "\n".join(c.get("value") or "" for c in entry.get("content", [])),
            "author": entry.get("author"),
            "tags": [t.get("term") for t in entry.get("tags", []) if t.get("term")],
        })

    return entries


def fetch_feed(url, use_cache=False, delay=REQUEST_DELAY):
    '''
    피드 요청 → {"url", "http_status", "error", "entries", "from_cache"}
    use_cache=True면 200/404 응답을 저장해두고 재요청하지 않음 (과거 날짜 아카이브 전용)
    '''
    path_ok = FEED_CACHE_DIR / f"{url_hash(url)}.xml.gz"
    path_404 = FEED_CACHE_DIR / f"{url_hash(url)}.404"

    if use_cache and path_ok.exists():
        content = gzip.decompress(path_ok.read_bytes())
        return {"url": url, "http_status": 200, "error": None, "entries": parse_feed(content), "from_cache": True}

    if use_cache and path_404.exists():
        return {"url": url, "http_status": 404, "error": None, "entries": [], "from_cache": True}

    response, error = polite_get(url, delay=delay)

    if response is None:
        return {"url": url, "http_status": None, "error": error, "entries": [], "from_cache": False}

    entries = parse_feed(response.content) if response.status_code == 200 else []

    if use_cache and response.status_code == 200:
        path_ok.write_bytes(gzip.compress(response.content))

    if use_cache and response.status_code == 404:
        path_404.touch()

    return {"url": url, "http_status": response.status_code, "error": None, "entries": entries, "from_cache": False}


def add_param(url, key, value):
    return f"{url}{'&' if '?' in url else '?'}{key}={value}"


def entry_times(entries):
    return [e["published_utc"] for e in entries if e["published_utc"]]

In [9]:
HTML_DIR.mkdir(parents=True, exist_ok=True)

HEADING_TAGS = {"h2", "h3", "h4", "h5"}
BLOCK_TAGS = {"p", "li", "blockquote", "pre"} | HEADING_TAGS
DROP_TAGS = ["script", "style", "noscript", "figure", "figcaption", "iframe", "video", "audio"]


def has_junk_class(node):
    classes = (node.get("class") or "").lower().split()

    return any(c.startswith(JUNK_CLASS_PREFIXES) for c in classes)


def html_to_text(fragment, drop_junk=False):
    # 본문 HTML 조각 → 문단 텍스트 (문단마다 줄바꿈). 사진·캡션·스크립트는 제거
    # drop_junk=True면 광고·뉴스레터·관련 기사 등 class가 붙은 요소도 제거 (사이트 본문 영역용)
    if not isinstance(fragment, str) or not fragment.strip():
        return ""

    try:
        tree = lxml_html.fragment_fromstring(fragment, create_parent="div")
    except Exception:
        return ""

    for node in tree.xpath(" | ".join(f".//{tag}" for tag in DROP_TAGS)):
        node.drop_tree()

    if drop_junk:
        for node in [n for n in tree.iter() if isinstance(n.tag, str) and has_junk_class(n)]:
            node.drop_tree()

    # 문단 안에 든 문단(li 안의 p 등)은 바깥 것만 사용. (태그, 텍스트)
    blocks = []

    for node in tree.iter(*BLOCK_TAGS):
        if any(parent.tag in BLOCK_TAGS for parent in node.iterancestors()):
            continue

        text = " ".join(node.text_content().split())

        if drop_junk and text:
            link_chars = sum(len(" ".join(a.text_content().split())) for a in node.iter("a"))

            if link_chars / len(text) >= MAX_LINK_DENSITY:
                continue

        if text:
            blocks.append((node.tag, text))

    # 문단 태그 없이 <br>로만 나뉜 본문
    if not blocks:
        return " ".join(tree.text_content().split())

    # 본문 영역 끝에 문단 없이 남은 소제목 제거 (9to5Mac "Accessory deals on Amazon" 아래 상품 링크가 빠지고 제목만 남는 경우)
    if drop_junk:
        while blocks and blocks[-1][0] in HEADING_TAGS:
            blocks.pop()

    return "\n".join(text for _, text in blocks)


def clean_rss_body(text):
    # RSS 본문 끝 출처 문구 제거
    lines = [line for line in text.split("\n") if not RSS_FOOTER_PATTERN.search(line.strip())]

    return "\n".join(lines).strip()


def rss_body_of(content):
    return clean_rss_body(html_to_text(content))


def is_teaser(text):
    return bool(text) and bool(RSS_TEASER_PATTERN.search(text[-300:]))


def word_count(text):
    return len((text or "").split())


def words_of(text):
    return re.findall(r"[a-z0-9]+", unicodedata.normalize("NFKC", text or "").lower())


def shingles(text, n=SHINGLE_SIZE):
    words = words_of(text)

    return {tuple(words[i:i + n]) for i in range(len(words) - n + 1)}


def share_in(part, whole, n=SHINGLE_SIZE):
    # part의 연속 n단어 묶음 중 whole에도 있는 비율 (part가 n단어보다 짧으면 None)
    part_set = shingles(part, n)

    if not part_set:
        return None

    return round(len(part_set & shingles(whole, n)) / len(part_set), 3)


def junk_hits(text):
    lines = [line.strip() for line in (text or "").split("\n")]

    return sorted(
        name
        for name, pattern in BODY_JUNK_PATTERNS.items()
        if any(pattern.search(line) for line in lines)
    )


def host_of(url):
    host = (urlparse(url or "").hostname or "").lower()

    return host[4:] if host.startswith("www.") else host


def is_excluded_section(url):
    host = host_of(url)
    target = host + urlparse(url).path
    rules = [
        rule
        for site, site_rules in SECTION_EXCLUDE_RULES.items()
        if host == site or host.endswith("." + site)
        for rule in site_rules
    ]

    return any(re.search(rule, target) for rule in rules)


def same_site(url_a, url_b):
    a, b = host_of(url_a), host_of(url_b)

    return a == b or a.endswith("." + b) or b.endswith("." + a)

In [10]:
def html_cache_path(url):
    return HTML_DIR / f"{url_hash(url)}.html.gz"


def load_fetch_log():
    # 같은 URL은 마지막 기록이 우선
    return {record["url"]: record for record in load_jsonl(HTML_FETCH_LOG_PATH)}


def is_retryable_fetch(record):
    # 일시적 오류(연결 오류, 429, 5xx)만 재시도 대상
    return record["fetch_status"] == "REQUEST_ERROR" or record["http_status"] in {429, 500, 502, 503, 504}


def fetch_html(url):
    response, error = polite_get(url, delay=BODY_FETCH_DELAY, headers=HTML_HEADERS, retries=BODY_FETCH_RETRIES)

    record = {
        "url": url,
        "final_url": None,
        "http_status": None,
        "content_type": None,
        "fetch_status": "REQUEST_ERROR",
        "error": error,
        "html_saved": False,
        "fetched_at": now_utc().isoformat(),
    }

    if response is not None:
        content_type = response.headers.get("Content-Type", "").lower()

        record.update(
            final_url=response.url,
            http_status=response.status_code,
            content_type=content_type,
            fetch_status=(
                "HTTP_ERROR" if response.status_code != 200
                else "NOT_HTML" if "html" not in content_type
                else "OK"
            ),
        )

        # 오류 응답이라도 HTML이면 저장 (차단 페이지 확인용)
        if "html" in content_type:
            html_cache_path(url).write_bytes(gzip.compress(response.content))
            record["html_saved"] = True

    append_jsonl(HTML_FETCH_LOG_PATH, record)

    return record


def interleave_by_host(urls):
    # 같은 사이트에 연속 요청하지 않도록 호스트를 번갈아 정렬 (호스트별 대기 시간이 겹치게)
    queues = {}

    for url in dict.fromkeys(urls):
        queues.setdefault(host_of(url), []).append(url)

    ordered = []
    queues = list(queues.values())

    while any(queues):
        for queue in queues:
            if queue:
                ordered.append(queue.pop(0))

    return ordered


def is_host_blocking(record):
    return record["fetch_status"] == "REQUEST_ERROR" or record["http_status"] in {401, 403, 429}


def fetch_html_cached(urls, retry_errors=False, verbose=True):
    log = load_fetch_log()
    urls = interleave_by_host(urls)

    todo = [
        url for url in urls
        if url not in log or (retry_errors and is_retryable_fetch(log[url]))
    ]

    if verbose:
        print(f"URL {len(urls)}개 중 캐시 사용 {len(urls) - len(todo)}개, 새로 다운로드 {len(todo)}개", flush=True)

    # 사이트별 연속 차단 횟수. 한도를 넘으면 그 사이트는 이번 실행에서 더 요청하지 않음 (캐시에 남기지 않음)
    fail_streak = {}
    skipped = {}

    for i, url in enumerate(todo, start=1):
        host = host_of(url)

        if fail_streak.get(host, 0) >= BODY_HOST_FAIL_LIMIT:
            log[url] = {
                "url": url, "final_url": None, "http_status": None, "content_type": None,
                "fetch_status": "SKIPPED_HOST", "error": f"연속 {BODY_HOST_FAIL_LIMIT}회 차단으로 건너뜀",
                "html_saved": False, "fetched_at": None,
            }
            skipped[host] = skipped.get(host, 0) + 1
            continue

        log[url] = fetch_html(url)
        fail_streak[host] = fail_streak.get(host, 0) + 1 if is_host_blocking(log[url]) else 0

        if verbose and (i % 50 == 0 or i == len(todo)):
            print(f"  [{i}/{len(todo)}] {host} {log[url]['fetch_status']} ({log[url]['http_status']})", flush=True)

    if verbose and skipped:
        print("연속 차단으로 건너뛴 사이트:", skipped)

    return {url: log[url] for url in urls}

In [11]:
def read_cached_html(url):
    path = html_cache_path(url)

    if not path.exists():
        return None

    return decode_file(gzip.decompress(path.read_bytes()))


def parse_json_ld(tree):
    # 페이지의 JSON-LD 최상위 객체(+@graph)를 평탄화해서 리스트로 반환
    items = []

    for node in tree.xpath('//script[@type="application/ld+json"]'):
        try:
            data = json.loads(node.text_content().strip())
        except Exception:
            continue

        stack = [data]

        while stack:
            current = stack.pop()

            if isinstance(current, list):
                stack.extend(current)

            elif isinstance(current, dict):
                items.append(current)

                if "@graph" in current:
                    stack.append(current["@graph"])

    return items


def ld_types_of(item):
    types = item.get("@type", [])

    if isinstance(types, str):
        types = [types]

    return {t for t in types if isinstance(t, str)}


def meta_content(tree, attr, value, lower=True):
    found = tree.xpath(f'//meta[@{attr}="{value}"]/@content')

    if not found:
        return None

    return found[0].strip().lower() if lower else found[0].strip()


def parse_bool(value):
    if isinstance(value, bool):
        return value

    if isinstance(value, str) and value.strip().lower() in {"true", "false"}:
        return value.strip().lower() == "true"

    return None


def to_news_tz(value):
    # 날짜 문자열 → 동부시간 datetime. 시간대 표기가 없으면 동부시간으로 간주
    if not isinstance(value, str) or not value.strip():
        return None

    try:
        dt = date_parser.parse(value.strip())
    except (ValueError, OverflowError):
        return None

    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=NEWS_TIMEZONE)

    return dt.astimezone(NEWS_TIMEZONE)


def find_page_date(tree, article_items, fields):
    # fields 순서대로 찾아서 처음 해석되는 날짜
    for field in fields:
        kind, key = field.split(":", 1)

        if kind == "ld":
            values = [item.get(key) for item in article_items]
        else:
            values = tree.xpath(f'//meta[@{kind}="{key}"]/@content')

        for value in values:
            dt = to_news_tz(value)

            if dt:
                return dt

    return None


def site_container_body(tree, final_url):
    # SITE_BODY_CONTAINERS에 등록된 본문 영역의 문단 텍스트 (여러 개면 문서 순서대로 이어붙임)
    container = SITE_BODY_CONTAINERS.get(host_of(final_url))

    if not container:
        return ""

    tag, class_name = container
    nodes = tree.xpath(f"//{tag}[contains(concat(' ', normalize-space(@class), ' '), ' {class_name} ')]")
    node_ids = {id(node) for node in nodes}

    # 영역 안에 같은 영역이 또 있으면 바깥 것만
    nodes = [node for node in nodes if not any(id(parent) in node_ids for parent in node.iterancestors())]

    texts = [html_to_text(lxml_html.tostring(node, encoding="unicode"), drop_junk=True) for node in nodes]

    return "\n".join(t for t in texts if t)


# WordPress 글 데이터(JSON)의 본문 HTML: "content":{"rendered":"..."}
EMBEDDED_CONTENT_PATTERN = re.compile(r'"content":\{"rendered":("(?:[^"\\]|\\.)*")')


def embedded_json_body(raw_html):
    texts = []

    for match in EMBEDDED_CONTENT_PATTERN.finditer(raw_html):
        try:
            texts.append(html_to_text(json.loads(match.group(1))))
        except ValueError:
            continue

    return max(texts, key=len, default="")


def extract_page_features(raw_html, final_url):
    # lxml은 encoding 선언이 있는 str을 받지 못하므로 XML 선언 제거
    tree = lxml_html.fromstring(re.sub(r"^\s*<\?xml[^>]*\?>", "", raw_html))

    page_title = (
        meta_content(tree, "property", "og:title", lower=False)
        or (tree.findtext(".//title") or "").strip()
    )

    og_type = meta_content(tree, "property", "og:type")

    ld_items = parse_json_ld(tree)
    ld_types = set().union(*[ld_types_of(item) for item in ld_items]) if ld_items else set()
    article_items = [item for item in ld_items if ld_types_of(item) & ARTICLE_LD_TYPES]

    # 언론사가 페이지에 넣어둔 기사 본문 (가장 긴 것)
    ld_body = max(
        (" ".join(html.unescape(str(item.get("articleBody") or "")).split()) for item in article_items),
        key=len,
        default="",
    )

    ld_free_values = [
        parse_bool(item.get("isAccessibleForFree"))
        for item in article_items
        if "isAccessibleForFree" in item
    ]
    ld_is_free = None if not ld_free_values else all(v is not False for v in ld_free_values)

    published_dt = find_page_date(tree, article_items, PUBLISHED_DATE_FIELDS)

    path = urlparse(final_url).path
    url_is_video = bool(VIDEO_PATH_PATTERN.search(path) or VIDEO_HOST_PATTERN.search(host_of(final_url)))
    url_is_live = bool(LIVE_BLOG_PATH_PATTERN.search(path))

    title_lower = page_title.lower()
    html_lower = raw_html[:50000].lower()
    is_blocked = (
        any(phrase in title_lower for phrase in BLOCK_TITLE_PHRASES)
        or any(marker in html_lower for marker in BLOCK_HTML_MARKERS)
    )

    trafilatura_body = trafilatura_extract(
        raw_html,
        url=final_url,
        include_comments=False,
        include_tables=False,
        favor_precision=True,
    )
    trafilatura_body = (trafilatura_body or "").strip()

    ld_words = word_count(ld_body)
    container_body = site_container_body(tree, final_url)

    # 교차 확인용 기준 본문: 언론사가 넣은 JSON-LD articleBody > 사이트별 본문 영역
    if ld_words >= MIN_BODY_WORDS:
        reference, reference_type = ld_body, "json_ld"
    elif word_count(container_body) >= MIN_BODY_WORDS:
        reference, reference_type = container_body, "container"
    else:
        reference, reference_type = "", None

    # 기준 본문 중 trafilatura 본문에 들어간 비율(누락 확인), trafilatura 본문 중 기준 본문에 있는 비율(오염 확인)
    ref_recall = share_in(reference, trafilatura_body) if reference and trafilatura_body else None
    ref_precision = share_in(trafilatura_body, reference) if reference and trafilatura_body else None

    body, body_method = trafilatura_body, "trafilatura"

    # 사이트별 본문 영역이 있으면 그것을 본문으로 사용 (trafilatura는 교차 확인용)
    #   trafilatura는 NY Post 구독 홍보·뉴스레터 박스·연합 기사 머리글, Ars Technica 사진 설명,
    #   9to5Mac 상품 목록까지 본문에 넣는 경우가 있었음 (2026-11-05 표본 380건 확인)
    if reference_type == "container":
        body, body_method = reference, reference_type

    # trafilatura가 JSON-LD 본문의 일부만 잡았으면 JSON-LD 본문 사용
    elif reference and (ref_recall or 0) < MIN_REF_COVERAGE and word_count(reference) > word_count(trafilatura_body):
        body, body_method = reference, reference_type

    # [2_rss_history_test.ipynb에 없는 규칙] trafilatura 본문에 JSON-LD 본문에 없는 내용이 10% 넘게 섞이면 JSON-LD 본문 사용
    #   LA Times: 제목·오디오 안내·관련 기사 소개·Perplexity AI 요약(Perspectives)이 trafilatura 본문에 섞임 (2026-11-05)
    elif reference_type == "json_ld" and (ref_precision or 0) < MIN_REF_PRECISION:
        body, body_method = reference, reference_type

    # 본문이 HTML 태그가 아니라 페이지 안 JSON에만 있는 경우 (National Review)
    elif word_count(body) < MIN_BODY_WORDS:
        embedded = embedded_json_body(raw_html)

        if word_count(embedded) >= MIN_BODY_WORDS:
            body, body_method = embedded, "embedded_json"

    body_words = word_count(body)
    body_lower = body.lower()

    return {
        "page_title": page_title,
        "og_type": og_type,
        "ld_types": sorted(ld_types),
        "ld_has_article": bool(ld_types & ARTICLE_LD_TYPES),
        "ld_has_video": "VideoObject" in ld_types,
        "ld_has_liveblog": "LiveBlogPosting" in ld_types,
        "ld_is_free": ld_is_free,
        "ld_body": ld_body,
        "ld_words": ld_words,
        "page_published_et": published_dt,
        "url_is_video": url_is_video,
        "url_is_live": url_is_live,
        "is_blocked": is_blocked,
        "paywall_phrase": next((p for p in PAYWALL_PHRASES if p in body_lower), None),
        "body": body,
        "body_method": body_method,
        "trafilatura_words": word_count(trafilatura_body),
        "container_words": word_count(container_body),
        "reference_type": reference_type,
        "ref_recall": ref_recall,
        "ref_precision": ref_precision,
        "body_words": body_words,
        "ld_coverage": round(body_words / ld_words, 3) if ld_words else None,
    }


def classify_page(link, fetch, features):
    # 제외 구역은 URL만으로 판단 (실제 수집에서는 다운로드 전에 거름)
    if is_excluded_section(link):
        return "EXCLUDED_SECTION"

    if fetch is None:
        return "NOT_FETCHED"

    if fetch["fetch_status"] in {"REQUEST_ERROR", "NOT_HTML", "SKIPPED_HOST"}:
        return fetch["fetch_status"]

    if features is not None and features["is_blocked"]:
        return "BLOCKED"

    if fetch["fetch_status"] == "HTTP_ERROR":
        return f"HTTP_{fetch['http_status']}"

    if not same_site(fetch["final_url"], link):
        return "REDIRECTED_AWAY"

    if features is None:
        return "NOT_FETCHED"

    if (
        features["url_is_video"]
        or (features["og_type"] or "").startswith("video")
        or (features["ld_has_video"] and not features["ld_has_article"])
    ):
        return "VIDEO_PAGE"

    if features["url_is_live"] or features["ld_has_liveblog"]:
        return "LIVE_BLOG"

    # 기사 타입 메타데이터(JSON-LD 기사 또는 og:type=article)가 있어야 기사로 인정
    # 1_google_rss_source_test.ipynb는 og:type≠website와 작성 시각 메타도 요구했지만, 여기 링크는 RSS 기사라 작성 시각을 이미 알고 있고
    #   - Mother Jones: JSON-LD·작성 시각 메타가 없는 정상 기사 (og:type=article)
    #   - WIRED 리뷰: og:type=website지만 JSON-LD Review인 정상 기사
    # 가 있어 뺐음 (작성 시각 차이는 date_gap_hours로 따로 확인)
    if not (features["ld_has_article"] or features["og_type"] == "article"):
        return "NOT_ARTICLE"

    if not features["body"]:
        return "EXTRACT_FAILED"

    if features["paywall_phrase"]:
        return "PAYWALL"

    body_is_short = features["body_words"] < MIN_BODY_WORDS
    body_is_partial = features["ld_coverage"] is not None and features["ld_coverage"] < MIN_LD_COVERAGE

    # isAccessibleForFree=false여도 HTML에 본문 전체가 있는 경우가 많아 본문이 짧거나 일부일 때만 유료로 판정
    if features["ld_is_free"] is False and (body_is_short or body_is_partial):
        return "PAYWALL"

    if body_is_short:
        return "TOO_SHORT"

    if body_is_partial:
        return "PARTIAL_BODY"

    return "SUCCESS"


def analyze_page(link, fetch):
    # 저장된 HTML로 (판정, 특징, 분석 오류)
    features = None
    error = None

    if fetch and fetch["html_saved"]:
        raw_html = read_cached_html(link)

        if raw_html:
            try:
                features = extract_page_features(raw_html, fetch["final_url"])
            except Exception as e:
                error = f"{type(e).__name__}: {e}"

    status = "ANALYZE_ERROR" if error else classify_page(link, fetch, features)

    return status, features, error

In [12]:
def sample_articles(articles_df, feeds, per_feed=BODY_SAMPLE_PER_FEED, seed=BODY_SAMPLE_SEED):
    groups = []

    for feed_id in feeds:
        group = articles_df[articles_df["feed_id"] == feed_id].drop_duplicates("link")
        groups.append(group.sample(min(per_feed, len(group)), random_state=seed))

    return pd.concat(groups, ignore_index=True)


def check_body(article, fetch):
    status, features, error = analyze_page(article["link"], fetch)
    features = features or {}

    page_body = features.get("body", "") if status == "SUCCESS" else ""
    rss_body = rss_body_of(article.get("content"))
    summary_text = html_to_text(article.get("summary")) or article.get("summary") or ""

    # RSS 본문이 원문 본문을 얼마나 덮는지 (원문 추출이 성공한 기사만 비교 가능)
    rss_recall = share_in(page_body, rss_body) if page_body and rss_body else None
    rss_full = rss_recall is not None and rss_recall >= RSS_FULL_COVERAGE

    if rss_full:
        body, body_source = rss_body, "RSS"
    elif page_body:
        body, body_source = page_body, "PAGE"
    else:
        body, body_source = "", None

    published = features.get("page_published_et")
    date_gap = (
        round((article["published_utc"] - published).total_seconds() / 3600, 2)
        if published is not None else None
    )

    return {
        "feed_id": article["feed_id"],
        "name": article["name"],
        "title": article["title"],
        "link": article["link"],
        "published_et": article["published_et"],
        "final_url": fetch["final_url"] if fetch else None,
        "http_status": fetch["http_status"] if fetch else None,
        "status": status,
        "analyze_error": error,
        "page_title": features.get("page_title"),
        "body_method": features.get("body_method"),
        "page_words": word_count(page_body),
        "trafilatura_words": features.get("trafilatura_words"),
        "ld_words": features.get("ld_words"),
        "container_words": features.get("container_words"),
        "reference_type": features.get("reference_type"),
        "ref_recall": features.get("ref_recall"),
        "ref_precision": features.get("ref_precision"),
        "rss_words": word_count(rss_body),
        "rss_teaser": is_teaser(rss_body),
        "rss_recall": rss_recall,
        "rss_precision": share_in(rss_body, page_body) if page_body and rss_body else None,
        "rss_full": rss_full,
        "title_match": share_in(article["title"], features.get("page_title") or "", n=1),
        "summary_in_body": share_in(summary_text, body, n=3) if body else None,
        "date_gap_hours": date_gap,
        "body_source": body_source,
        # 최종 본문을 어디서 얻었는지: rss / trafilatura / json_ld / container / embedded_json
        "final_method": "rss" if body_source == "RSS" else features.get("body_method") if body_source == "PAGE" else None,
        "body_words": word_count(body),
        "junk": junk_hits(body),
        "body": body,
        "rss_body": rss_body,
        "page_body": page_body,
    }

In [13]:
def summarize_bodies(check_df):
    df = check_df.assign(
        is_article=lambda d: ~d["status"].isin(FILTERABLE_STATUSES),
        is_success=lambda d: d["status"] == "SUCCESS",
        has_rss_body=lambda d: d["rss_words"] >= MIN_BODY_WORDS,
        rss_comparable=lambda d: d["rss_recall"].notna(),
        has_body=lambda d: d["body_words"] > 0,
        has_junk=lambda d: d["junk"].str.len() > 0,
        title_mismatch=lambda d: d["title_match"] < 0.5,
        date_gap_over_24h=lambda d: d["date_gap_hours"].abs() > 24,
    )
    articles = df[df["is_article"]]

    summary = df.groupby("feed_id").agg(
        name=("name", "first"),
        sampled=("link", "size"),
        rss_body_rate=("has_rss_body", "mean"),
        rss_teaser_rate=("rss_teaser", "mean"),
        rss_comparable=("rss_comparable", "sum"),
        rss_recall_med=("rss_recall", "median"),
        title_match_med=("title_match", "median"),
        title_mismatch=("title_mismatch", "sum"),
        summary_in_body_med=("summary_in_body", "median"),
        date_gap_over_24h=("date_gap_over_24h", "sum"),
    ).join(
        articles.groupby("feed_id").agg(
            article_pages=("link", "size"),
            page_success_rate=("is_success", "mean"),
            page_words_med=("page_words", lambda x: x[x > 0].median()),
            ref_compared=("ref_recall", "count"),
            ref_recall_med=("ref_recall", "median"),
            ref_precision_med=("ref_precision", "median"),
            body_ok_rate=("has_body", "mean"),
            junk_rate=("has_junk", lambda x: x[articles.loc[x.index, "has_body"]].mean()),
        )
    )

    summary["rss_full_rate"] = (
        df[df["rss_comparable"]].groupby("feed_id")["rss_full"].mean()
    )

    failures = articles[articles["status"] != "SUCCESS"]
    summary["main_fail_reason"] = failures.groupby("feed_id")["status"].agg(lambda x: x.value_counts().index[0])

    def body_verdict(row):
        if not row["article_pages"] >= MIN_BODY_ARTICLES:
            return "INSUFFICIENT_SAMPLE"

        if row["rss_comparable"] >= MIN_RSS_COMPARABLE and row["rss_full_rate"] >= BODY_USABLE_RATE:
            return "RSS_BODY"

        if row["page_success_rate"] >= BODY_USABLE_RATE:
            return "PAGE_BODY"

        if row["page_success_rate"] >= BODY_UNSTABLE_RATE:
            return "UNSTABLE"

        return "NOT_USABLE"

    summary["body_verdict"] = summary.apply(body_verdict, axis=1)

    return summary.reset_index()

## 1. 후보 피드 + 팀원 수집기 대조

과거 수집 테스트의 `LIVE_ONLY` 피드 전부가 후보입니다.
팀원 통합 수집기(`rss_pipeline.load_feeds()`)가 이미 받는 URL과 같은 피드는 `in_team_pipeline=True`로 표시합니다.

In [14]:
def normalize_feed_url(url):
    parsed = urlparse(url)
    host = (parsed.hostname or "").lower()
    host = host[4:] if host.startswith("www.") else host

    return f"{host}{parsed.path.rstrip('/')}?{parsed.query}"


def load_team_feeds():
    # 팀원 수집기 설정을 그대로 읽음 (네트워크 요청 없음)
    import sys

    src_dir = str((TEAM_RSS_DIR / "src").resolve())

    if src_dir not in sys.path:
        sys.path.insert(0, src_dir)

    import rss_pipeline

    return rss_pipeline.load_feeds(config_dir=TEAM_RSS_DIR / "config")


past_lists = json.loads(PAST_FEED_LISTS_PATH.read_text())
LIVE_CANDIDATE_FEEDS = {feed_id: CANDIDATE_FEEDS[feed_id] for feed_id in past_lists["LIVE_ONLY"]}

team_feeds = load_team_feeds()
TEAM_FEED_URLS = {normalize_feed_url(f["feed_url"]): f for f in team_feeds}
TEAM_SOURCE_NAMES = sorted({f["source_name"] for f in team_feeds})

candidate_df = pd.DataFrame([
    {
        "feed_id": feed_id,
        "source": info["source"],
        "name": info["name"],
        "section": info["section"],
        "tier": tier_of(info)[0],
        "tier_reason": tier_of(info)[1],
        "in_team_pipeline": normalize_feed_url(info["url"]) in TEAM_FEED_URLS,
        "url": info["url"],
    }
    for feed_id, info in LIVE_CANDIDATE_FEEDS.items()
])

print(f"후보 {len(candidate_df)}개 피드 / 팀원 수집기 {len(TEAM_SOURCE_NAMES)}개 매체·{len(team_feeds)}개 피드")
print("팀원 수집기에 이미 있는 피드:", candidate_df.loc[candidate_df["in_team_pipeline"], "feed_id"].tolist())

pd.crosstab(candidate_df["tier"], candidate_df["in_team_pipeline"], margins=True)

후보 60개 피드 / 팀원 수집기 19개 매체·36개 피드
팀원 수집기에 이미 있는 피드: ['cnbc_top', 'cnbc_business', 'cnbc_economy', 'cnbc_finance', 'cnbc_tech', 'fortune', 'business_insider', 'foxbusiness_latest', 'foxbusiness_markets', 'foxbusiness_economy', 'cbs_moneywatch', 'npr_business', 'engadget']


in_team_pipeline,False,True,All
tier,,,
EXCLUDE,14,0,14
FINANCE_CORE,15,10,25
GENERAL_MAJOR,13,2,15
PRESS_RELEASE,2,0,2
TECH,3,1,4
All,47,13,60


## 2. 현재 RSS 응답 + 갱신 속도

피드를 지금 받아서 다음을 계산합니다.
- `per_day`: 하루 발행량. 최근 24시간 기사 수로 계산하되, 피드 기사가 모두 24시간 안이면 피드 맨 앞~맨 뒤 기사 사이 시간으로 계산
- `cover_hours`: 피드 하나가 담는 시간 = 피드 기사 수 ÷ 시간당 발행량. 이 시간 안에 다시 받지 않으면 밀려난 기사를 놓침
- `poll_minutes`: 권장 수집 간격 = `cover_hours` × 0.5 (앞뒤가 겹치도록), 5분~6시간으로 제한
- `rss_body_words`: RSS 본문(`content`) 단어 수 중앙값 (0이면 요약만 제공)

피드 첫 화면 기사 날짜 분포(`span_hours`)는 오래된 기사가 섞이면 크게 왜곡돼서(예: Guardian 1건이 12년 전) 쓰지 않습니다.

In [15]:
def check_live_rate(feed_id, info, now):
    result = fetch_feed(info["url"])
    entries = result["entries"]
    times = entry_times(entries)

    row = {
        "feed_id": feed_id,
        "http_status": result["http_status"],
        "error": result["error"],
        "entries": len(entries),
        "newest_et": max(times).astimezone(NEWS_TIMEZONE) if times else None,
    }

    if not entries:
        return {**row, "live_status": "EMPTY" if result["http_status"] == 200 else "HTTP_ERROR"}

    newest_age_hours = (now - max(times)).total_seconds() / 3600 if times else None
    recent = sum(1 for t in times if (now - t).total_seconds() <= RATE_WINDOW_HOURS * 3600)

    # 피드 기사가 모두 최근 24시간 안이면 24시간 기사 수가 피드 크기에 막혀 과소 계산됨
    # → 피드 맨 앞~맨 뒤 기사 사이 시간으로 계산 (Investing.com: 10건이 6분 사이)
    if times and len(times) >= 2 and (now - min(times)).total_seconds() <= RATE_WINDOW_HOURS * 3600:
        span_hours = max((max(times) - min(times)).total_seconds() / 3600, 1 / 60)
        per_hour = (len(times) - 1) / span_hours
    else:
        per_hour = recent / RATE_WINDOW_HOURS

    if newest_age_hours is None or newest_age_hours > STALE_DAYS * 24:
        status = "STALE"
    else:
        status = "OK"

    # 하루 기사가 0~1건이면 발행량이 적어 최대 간격으로 충분
    cover_hours = len(entries) / per_hour if per_hour > 0 else None
    poll_minutes = (
        MAX_POLL_MINUTES if cover_hours is None
        else min(MAX_POLL_MINUTES, max(MIN_POLL_MINUTES, cover_hours * 60 * POLL_SAFETY))
    )

    rss_words = [word_count(rss_body_of(e["content"])) for e in entries]

    return {
        **row,
        "live_status": status,
        "newest_age_hours": round(newest_age_hours, 1) if newest_age_hours is not None else None,
        "per_day": round(per_hour * 24, 1),
        "cover_hours": round(cover_hours, 1) if cover_hours else None,
        "poll_minutes": round(poll_minutes),
        "miss_risk": cover_hours is not None and cover_hours * 60 * POLL_SAFETY < MIN_POLL_MINUTES,
        "rss_body_words": float(pd.Series(rss_words).median()),
        "entries_list": entries,
    }


now = now_utc()
live_rows = []

for feed_id, info in LIVE_CANDIDATE_FEEDS.items():
    live_rows.append(check_live_rate(feed_id, info, now))

live_df = candidate_df.merge(pd.DataFrame(live_rows), on="feed_id")
live_df.drop(columns="entries_list").to_csv(LIVE_CHECK_PATH, index=False, encoding="utf-8-sig")

pd.set_option("display.width", 250)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 30)

live_df[[
    "feed_id", "tier", "in_team_pipeline", "live_status", "entries", "newest_age_hours",
    "per_day", "cover_hours", "poll_minutes", "miss_risk", "rss_body_words",
]].sort_values(["tier", "feed_id"])

,feed_id,tier,in_team_pipeline,live_status,entries,newest_age_hours,per_day,cover_hours,poll_minutes,miss_risk,rss_body_words
55,cnet,EXCLUDE,False,OK,25,1.1,14.0,42.9,360,False,0.0
20,fool,EXCLUDE,False,OK,50,0.2,159.8,7.5,225,False,119.5
34,forbes_business,EXCLUDE,False,OK,25,0.7,46.6,12.9,360,False,25.0
35,forbes_innovation,EXCLUDE,False,OK,25,0.7,35.6,16.9,360,False,24.0
26,kiplinger,EXCLUDE,False,OK,50,0.7,6.0,200.0,360,False,1129.5
24,marketbeat,EXCLUDE,False,OK,100,22.3,3.0,800.0,360,False,0.0
21,nasdaq_markets,EXCLUDE,False,OK,15,0.5,409.1,0.9,26,False,0.0
33,nypost_business,EXCLUDE,False,OK,20,1.7,25.0,19.2,360,False,25.0
36,qz,EXCLUDE,False,OK,5,1.5,2832.8,0.0,5,True,0.0
18,seekingalpha_all,EXCLUDE,False,OK,30,0.1,700.9,1.0,31,False,0.0


## 3. 본문 확보 테스트 (현재 기사)

수집 등급(`COLLECT_TIERS`)이면서 **팀원 수집기에 없는 언론사**만 대상입니다. 팀원 수집기 매체는 팀원이 매체별 규칙으로 이미 검증했습니다.
언론사마다 현재 피드 기사 중 최대 8건을 뽑아 `2_rss_history_test.ipynb` 7단계와 같은 방식으로 원문을 받고 RSS 본문과 대조합니다.

In [16]:
def live_sample(live_df, per_source=LIVE_SAMPLE_PER_SOURCE, seed=BODY_SAMPLE_SEED):
    rows = []
    targets = live_df[
        live_df["tier"].isin(COLLECT_TIERS)
        & (live_df["live_status"] == "OK")
        & ~live_df["name"].isin(TEAM_SOURCE_NAMES)
    ]

    for source, group in targets.groupby("source"):
        articles = []

        for row in group.to_dict("records"):
            for entry in row["entries_list"]:
                # 다른 사이트로 연결되는 링크(통신 전재 등)는 제외
                if entry["link"] and same_site(entry["link"], "https://" + source):
                    articles.append({"feed_id": source, "name": row["name"], **entry})

        df = pd.DataFrame(articles).drop_duplicates("link")

        if len(df):
            rows.append(df.sample(min(per_source, len(df)), random_state=seed))

    return pd.concat(rows, ignore_index=True)


live_sample_df = live_sample(live_df)
print(f"표본 {len(live_sample_df)}건 / 언론사 {live_sample_df['feed_id'].nunique()}곳")

live_fetch_map = fetch_html_cached(
    [link for link in live_sample_df["link"] if not is_excluded_section(link)],
    retry_errors=True,
)

live_body_df = pd.DataFrame([
    check_body(article, live_fetch_map.get(article["link"]))
    for article in live_sample_df.to_dict("records")
])
live_body_df.to_parquet(BODY_CHECK_PATH, index=False)

pd.crosstab(live_body_df["feed_id"], live_body_df["status"], margins=True)

표본 119건 / 언론사 15곳
URL 119개 중 캐시 사용 66개, 새로 다운로드 53개


  [53/53] wsj.com HTTP_ERROR (401)


연속 차단으로 건너뛴 사이트: {'bloomberg.com': 4, 'axios.com': 2, 'businesswire.com': 2, 'investing.com': 2, 'marketwatch.com': 2, 'nytimes.com': 2, 'politico.com': 2}


status,BLOCKED,HTTP_403,PAYWALL,SKIPPED_HOST,SUCCESS,All
feed_id,,,,,,
axios.com,6,0,0,2,0,8
bloomberg.com,4,0,0,4,0,8
businesswire.com,6,0,0,2,0,8
investing.com,0,6,0,2,0,8
latimes.com,0,0,0,0,8,8
marketwatch.com,6,0,0,2,0,8
nytimes.com,6,0,0,2,0,8
politico.com,6,0,0,2,0,8
prnewswire.com,0,0,0,0,8,8


In [17]:
live_body_summary = summarize_bodies(live_body_df)
live_body_summary.to_csv(BODY_SUMMARY_PATH, index=False, encoding="utf-8-sig")

live_body_summary[[
    "feed_id", "body_verdict", "sampled", "article_pages", "page_success_rate", "page_words_med",
    "ref_compared", "ref_recall_med", "ref_precision_med",
    "rss_body_rate", "rss_comparable", "rss_full_rate",
    "title_match_med", "junk_rate", "main_fail_reason",
]].round(3).sort_values(["body_verdict", "feed_id"])

,feed_id,body_verdict,sampled,article_pages,page_success_rate,page_words_med,ref_compared,ref_recall_med,ref_precision_med,rss_body_rate,rss_comparable,rss_full_rate,title_match_med,junk_rate,main_fail_reason
0,axios.com,NOT_USABLE,8,8,0.0,NaN,0,NaN,NaN,1.00,0,NaN,0.0,NaN,BLOCKED
1,bloomberg.com,NOT_USABLE,8,8,0.0,NaN,0,NaN,NaN,0.00,0,NaN,0.0,NaN,BLOCKED
2,businesswire.com,NOT_USABLE,8,8,0.0,NaN,0,NaN,NaN,0.00,0,NaN,0.0,NaN,BLOCKED
3,investing.com,NOT_USABLE,8,8,0.0,NaN,0,NaN,NaN,0.00,0,NaN,0.0,NaN,HTTP_403
5,marketwatch.com,NOT_USABLE,8,8,0.0,NaN,0,NaN,NaN,0.00,0,NaN,0.0,NaN,BLOCKED
6,nytimes.com,NOT_USABLE,8,8,0.0,NaN,0,NaN,NaN,0.00,0,NaN,0.0,NaN,BLOCKED
7,politico.com,NOT_USABLE,8,8,0.0,NaN,0,NaN,NaN,0.75,0,NaN,0.0,NaN,BLOCKED
14,wsj.com,NOT_USABLE,8,8,0.0,NaN,0,NaN,NaN,0.00,0,NaN,0.0,NaN,BLOCKED
4,latimes.com,PAGE_BODY,8,8,1.0,545.5,8,0.998,0.920,0.00,8,0.00,1.0,0.00,NaN
8,prnewswire.com,PAGE_BODY,8,8,1.0,1611.5,0,NaN,NaN,0.00,0,NaN,1.0,0.00,NaN


In [18]:
# 최종 본문 출처, 오염 문구
display(pd.crosstab(live_body_df["feed_id"], live_body_df["final_method"].fillna("(본문 없음)"), margins=True))

(
    live_body_df[live_body_df["body_words"] > 0]
    .explode("junk")
    .dropna(subset=["junk"])
    .pivot_table(index="feed_id", columns="junk", values="link", aggfunc="count", fill_value=0)
)

final_method,(본문 없음),json_ld,rss,trafilatura,All
feed_id,,,,,
axios.com,8,0,0,0,8
bloomberg.com,8,0,0,0,8
businesswire.com,8,0,0,0,8
investing.com,8,0,0,0,8
latimes.com,0,3,0,5,8
marketwatch.com,8,0,0,0,8
nytimes.com,8,0,0,0,8
politico.com,8,0,0,0,8
prnewswire.com,0,0,0,8,8


junk,related
feed_id,
time.com,2


## 4. 최종 판정 + 변수 정리

| 변수 | 조건 |
|---|---|
| `LIVE_COLLECT_FEEDS` | 수집 등급 + RSS 정상 + 본문 `RSS_BODY`/`PAGE_BODY` (팀원 수집기에 없는 것) |
| `LIVE_UNSTABLE_FEEDS` | 수집 등급 + RSS 정상 + 본문 `UNSTABLE` |
| `LIVE_RSS_UNVERIFIED_FEEDS` | 원문은 막혔지만 RSS 본문이 80단어 이상인 기사가 80% 이상, 티저 20% 이하 → RSS 본문 사용 가능성 (원문 대조 불가) |
| `LIVE_LIST_ONLY_FEEDS` | 수집 등급 + RSS 정상이지만 본문 불가 → 제목·요약만 (팀 기준상 분석에는 쓰지 않음) |
| `LIVE_EXCLUDED_FEEDS` | `EXCLUDE` 등급 또는 RSS 응답 이상 |
| `IN_TEAM_PIPELINE_FEEDS` | 팀원 수집기에 이미 있는 피드 (참고: 권장 수집 간격) |

각 값에 `poll_minutes`(권장 수집 간격)를 넣어둡니다.

In [19]:
BODY_VERDICT_BY_SOURCE = live_body_summary.set_index("feed_id")["body_verdict"].to_dict()
RSS_ONLY_STATS = live_body_summary.set_index("feed_id")[["rss_body_rate", "rss_teaser_rate"]].to_dict(orient="index")

live_df["body_verdict"] = live_df["source"].map(BODY_VERDICT_BY_SOURCE)
live_df.loc[live_df["in_team_pipeline"], "body_verdict"] = "TEAM_PIPELINE"

# 같은 언론사의 다른 섹션이 팀원 수집기에 있으면 팀원 매체별 본문 규칙을 그대로 쓰면 됨
live_df.loc[live_df["name"].isin(TEAM_SOURCE_NAMES) & ~live_df["in_team_pipeline"], "body_verdict"] = "TEAM_RULES"


def live_decision(row):
    if row["in_team_pipeline"]:
        return "IN_TEAM_PIPELINE"

    if row["tier"] not in COLLECT_TIERS:
        return "EXCLUDED"

    if row["live_status"] != "OK":
        return "EXCLUDED"

    # 같은 언론사가 팀원 수집기에 다른 섹션으로 있으면 팀원 규칙으로 본문 처리 (섹션 피드만 추가)
    if row["name"] in TEAM_SOURCE_NAMES:
        return "COLLECT"

    decision = {
        "RSS_BODY": "COLLECT",
        "PAGE_BODY": "COLLECT",
        "UNSTABLE": "UNSTABLE",
    }.get(row["body_verdict"], "LIST_ONLY")

    # 원문은 막혔지만 RSS에 긴 본문이 있는 경우: 원문과 대조할 수 없어 따로 분류
    stats = RSS_ONLY_STATS.get(row["source"])

    if decision == "LIST_ONLY" and stats and stats["rss_body_rate"] >= RSS_ONLY_MIN_RATE and stats["rss_teaser_rate"] <= RSS_ONLY_MAX_TEASER:
        return "RSS_UNVERIFIED"

    return decision


live_df["decision"] = live_df.apply(live_decision, axis=1)


def exclude_reason(row):
    if row["tier"] not in COLLECT_TIERS:
        return f"{row['tier']}: {row['tier_reason']}"

    return f"RSS {row['live_status']}"


def feeds_by_decision(decision):
    rows = live_df[live_df["decision"] == decision]

    return {
        row["feed_id"]: {
            **CANDIDATE_FEEDS[row["feed_id"]],
            "tier": row["tier"],
            "poll_minutes": row["poll_minutes"],
            "body_verdict": row["body_verdict"],
            **({"exclude_reason": exclude_reason(row)} if decision == "EXCLUDED" else {}),
        }
        for row in rows.to_dict("records")
    }


LIVE_COLLECT_FEEDS = feeds_by_decision("COLLECT")
LIVE_UNSTABLE_FEEDS = feeds_by_decision("UNSTABLE")
LIVE_RSS_UNVERIFIED_FEEDS = feeds_by_decision("RSS_UNVERIFIED")
LIVE_LIST_ONLY_FEEDS = feeds_by_decision("LIST_ONLY")
LIVE_EXCLUDED_FEEDS = feeds_by_decision("EXCLUDED")
IN_TEAM_PIPELINE_FEEDS = feeds_by_decision("IN_TEAM_PIPELINE")

live_df.drop(columns="entries_list").to_csv(LIVE_SUMMARY_PATH, index=False, encoding="utf-8-sig")

LIVE_LISTS_PATH.write_text(json.dumps({
    "created_at": now_utc().isoformat(),
    "COLLECT": LIVE_COLLECT_FEEDS,
    "UNSTABLE": LIVE_UNSTABLE_FEEDS,
    "RSS_UNVERIFIED": LIVE_RSS_UNVERIFIED_FEEDS,
    "LIST_ONLY": LIVE_LIST_ONLY_FEEDS,
    "EXCLUDED": LIVE_EXCLUDED_FEEDS,
    "IN_TEAM_PIPELINE": IN_TEAM_PIPELINE_FEEDS,
}, ensure_ascii=False, indent=2, default=str))

for name, feeds in [
    ("COLLECT", LIVE_COLLECT_FEEDS), ("UNSTABLE", LIVE_UNSTABLE_FEEDS),
    ("RSS_UNVERIFIED", LIVE_RSS_UNVERIFIED_FEEDS), ("LIST_ONLY", LIVE_LIST_ONLY_FEEDS),
    ("EXCLUDED", LIVE_EXCLUDED_FEEDS), ("IN_TEAM_PIPELINE", IN_TEAM_PIPELINE_FEEDS),
]:
    print(f"{name:17s} {len(feeds):2d}개: {list(feeds)}")

live_df[["feed_id", "name", "tier", "decision", "body_verdict", "poll_minutes", "per_day"]].sort_values(["decision", "tier", "feed_id"])

COLLECT           14개: ['cnbc_investing', 'cnbc_market_insider', 'cnbc_earnings', 'semafor', 'latimes_business', 'guardian_us_business', 'npr_economy', 'abc_business', 'nbc_business', 'time', 'theverge', 'tomshardware', 'venturebeat', 'prnewswire']
UNSTABLE           0개: []
RSS_UNVERIFIED     1개: ['axios']
LIST_ONLY         18개: ['wsj_markets', 'wsj_business', 'wsj_economy', 'wsj_tech', 'wsj_world', 'marketwatch_top', 'marketwatch_bulletins', 'bloomberg_markets', 'bloomberg_economics', 'bloomberg_tech', 'investing_news', 'investing_economy', 'nyt_business', 'nyt_economy', 'nyt_tech', 'nyt_home', 'politico', 'businesswire']
EXCLUDED          14개: ['seekingalpha_all', 'seekingalpha_currents', 'fool', 'nasdaq_markets', 'marketbeat', 'thestreet', 'kiplinger', 'nypost_business', 'forbes_business', 'forbes_innovation', 'qz', 'yahoo_news', 'washington_examiner', 'cnet']
IN_TEAM_PIPELINE  13개: ['cnbc_top', 'cnbc_business', 'cnbc_economy', 'cnbc_finance', 'cnbc_tech', 'fortune', 'business_insid

,feed_id,name,tier,decision,body_verdict,poll_minutes,per_day
5,cnbc_earnings,CNBC,FINANCE_CORE,COLLECT,TEAM_RULES,360,13.0
2,cnbc_investing,CNBC,FINANCE_CORE,COLLECT,TEAM_RULES,360,9.0
3,cnbc_market_insider,CNBC,FINANCE_CORE,COLLECT,TEAM_RULES,360,3.0
47,abc_business,ABC News,GENERAL_MAJOR,COLLECT,TEAM_RULES,360,8.0
44,guardian_us_business,The Guardian,GENERAL_MAJOR,COLLECT,TEAM_RULES,360,3.0
43,latimes_business,Los Angeles Times,GENERAL_MAJOR,COLLECT,PAGE_BODY,360,5.0
48,nbc_business,NBC News,GENERAL_MAJOR,COLLECT,TEAM_RULES,360,3.0
46,npr_economy,NPR,GENERAL_MAJOR,COLLECT,TEAM_RULES,360,1.0
38,semafor,Semafor,GENERAL_MAJOR,COLLECT,RSS_BODY,360,35.0
51,time,TIME,GENERAL_MAJOR,COLLECT,RSS_BODY,360,14.0


In [20]:
# 팀원 수집기 설정(feed_registry.csv) 형식으로 추가 후보 저장
registry_df = pd.DataFrame([
    {
        "feed_id": feed_id,
        "source_name": info["name"],
        "feed_name": info["section"].lower().replace(" ", "_"),
        "feed_url": info["url"],
        "collection_tier": "test",
        "enabled": "true",
        "last_verified_date": now_utc().date().isoformat(),
        "last_feed_status": "ok",
        "category": info["tier"].lower(),
        "body_quality_status": "team_rules" if info["body_verdict"] == "TEAM_RULES" else "unverified",
        "body_collection_enabled": "true",
        "notes": f"newsrss_live_test: body={info['body_verdict']}, poll≈{info['poll_minutes']}min",
    }
    for feed_id, info in {**LIVE_COLLECT_FEEDS, **LIVE_UNSTABLE_FEEDS}.items()
])
registry_df.to_csv(REGISTRY_EXPORT_PATH, index=False, encoding="utf-8-sig")

print(f"저장: {REGISTRY_EXPORT_PATH} ({len(registry_df)}개 피드)")
registry_df[["feed_id", "source_name", "feed_name", "category", "notes"]]

저장: newsrss_live/registry_additions.csv (14개 피드)


,feed_id,source_name,feed_name,category,notes
0,cnbc_investing,CNBC,investing,finance_core,"newsrss_live_test: body=TEAM_RULES, poll≈360min"
1,cnbc_market_insider,CNBC,market_insider,finance_core,"newsrss_live_test: body=TEAM_RULES, poll≈360min"
2,cnbc_earnings,CNBC,earnings,finance_core,"newsrss_live_test: body=TEAM_RULES, poll≈360min"
3,semafor,Semafor,all,general_major,"newsrss_live_test: body=RSS_BODY, poll≈360min"
4,latimes_business,Los Angeles Times,business,general_major,"newsrss_live_test: body=PAGE_BODY, poll≈360min"
5,guardian_us_business,The Guardian,us_business,general_major,"newsrss_live_test: body=TEAM_RULES, poll≈360min"
6,npr_economy,NPR,economy,general_major,"newsrss_live_test: body=TEAM_RULES, poll≈360min"
7,abc_business,ABC News,business,general_major,"newsrss_live_test: body=TEAM_RULES, poll≈360min"
8,nbc_business,NBC News,business,general_major,"newsrss_live_test: body=TEAM_RULES, poll≈360min"
9,time,TIME,all,general_major,"newsrss_live_test: body=RSS_BODY, poll≈360min"


## 5. 본문 눈으로 검수하기

In [21]:
def show_live_bodies(source, status="SUCCESS", n=1, chars=300):
    rows = live_body_df[(live_body_df["feed_id"] == source) & (live_body_df["status"] == status)]

    for row in rows.head(n).to_dict("records"):
        body = row["body"] or ""
        print(f"[{row['status']} / {row['final_method']}] {row['title']}")
        print(f"URL: {row['final_url'] or row['link']}")
        print(
            f"원문 {row['page_words']}단어 | trafilatura {row['trafilatura_words']} | JSON-LD {row['ld_words']} "
            f"| 기준 대비 recall {row['ref_recall']} precision {row['ref_precision']} | RSS {row['rss_words']}단어 | 오염 {row['junk']}"
        )
        print("── 처음 ──")
        print(body[:chars])
        print("── 끝 ──")
        print(body[-chars:] if len(body) > chars else "")
        print("\n" + "=" * 100 + "\n")


for source in sorted(live_body_df["feed_id"].unique()):
    show_live_bodies(source)

[SUCCESS / trafilatura] The millions spent by California billionaires against Prop. 40 is about 0.07% of their wealth
URL: https://www.latimes.com/politics/story/2026-10-23/these-californians-spent-0-07-of-their-billions-to-oppose-prop-40
원문 1113단어 | trafilatura 1113.0 | JSON-LD 1008.0 | 기준 대비 recall 1.0 precision 0.937 | RSS 18단어 | 오염 []
── 처음 ──
The millions spent by California billionaires against Prop. 40 is about 0.07% of their wealth
- Click here to listen to this article
- 1 in 8 California billionaires spent money opposing Proposition 40.
- The four richest Californians would owe 40% of the estimated $107 billion in taxes paid from th
── 끝 ──
dical Assn., and Planned Parenthood Affiliates of California. Supporters include Service Employees International Union, U.S. Sen. Bernie Sanders (I-Vt.), the California Democratic Party and the California Federation of Labor Unions AFL-CIO.
Times staff writer Queenie Wong contributed to this report.


[SUCCESS / trafilatura] Trimble Integr

In [22]:
# 실패 기사 (언론사별 주 실패 사유 1건)
for source, reason in live_body_summary.dropna(subset=["main_fail_reason"]).set_index("feed_id")["main_fail_reason"].items():
    show_live_bodies(source, status=reason)

[BLOCKED / None] Sen. Chris Murphy hospitalized after suffering stroke
URL: https://www.axios.com/2026/11/01/chris-murphy-stroke-hospitalized
원문 0단어 | trafilatura 0.0 | JSON-LD 0.0 | 기준 대비 recall nan precision nan | RSS 218단어 | 오염 []
── 처음 ──

── 끝 ──



[BLOCKED / None] Treasuries Gain as US Vote Fuels Hopes for Curbing Bond Supply
URL: https://www.bloomberg.com/news/articles/2026-11-05/treasuries-gain-as-us-vote-fuels-hopes-for-curbing-bond-supply
원문 0단어 | trafilatura 79.0 | JSON-LD 0.0 | 기준 대비 recall nan precision nan | RSS 0단어 | 오염 []
── 처음 ──

── 끝 ──



[BLOCKED / None] Samenvatting: Gencom neemt Rosewood London over en breidt zijn aanwezigheid uit met zijn eerste acquisitie van een luxehotel in Europa
URL: http://www.businesswire.com/news/home/20261103926361/nl/?feedref=JjAwJuNHiystnCoBq_hl-Rc4vIAVcHHkbDcwJimU8QtrtlakeQ9hNboBqTAWIjTge3KWq9s9jif-UkBjBsFRyYAbRTSLTc1mgvhPlnaBA55M-oupQnbXnhKsYk8RmHF_kAy2gZikaX3QWV6xOvgFlA==
원문 0단어 | trafilatura 15.0 | JSON-LD 0.0 | 기준 대비 recall nan 